# Brain Tumor Classification: Decision Tree Rule Extraction

This notebook:
1. Loads cached image features (tags + GLCM/statistical features) for the brain tumor dataset
2. Trains a shallow decision tree (max depth 4)
3. Extracts interpretable rules and selects the two highest-coverage, high-purity rules per class

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from services import image_service

## 2. Load Cached Image Features

In [ ]:
DATASET_PATH = '../input/brain_tumor_images'

# Load image paths and labels from folder structure
image_paths, labels, class_names = image_service.load_image_dataset(DATASET_PATH)
print(f'Classes: {class_names}')
print(f'Total images: {len(image_paths)}')
print(f'  Positive (tumor): {sum(labels)}')
print(f'  Negative (no tumor): {len(labels) - sum(labels)}')

In [ ]:
# Load cached tags (no OpenAI key needed - uses existing cache)
tags_cache = image_service.extract_and_cache_tags(
    DATASET_PATH, image_paths, openai_api_key=None, force_refresh=False
)
print(f'Tags cached for {len(tags_cache)} images')

# Load cached image features
features_cache, feature_names = image_service.extract_and_cache_image_features(
    DATASET_PATH, image_paths, force_refresh=False
)
print(f'Image features cached for {len(features_cache)} images')
print(f'Feature names ({len(feature_names)}): {feature_names}')

In [ ]:
# Create tabular dataset from caches
df, all_tags, all_feature_cols = image_service.create_tabular_dataset(
    DATASET_PATH, image_paths, labels,
    tags_cache, features_cache, feature_names
)

print(f'Tabular dataset shape: {df.shape}')
print(f'Tag features ({len(all_tags)}): {all_tags[:10]}...')
print(f'All feature columns: {len(all_feature_cols)}')
df.head()

## 3. Prepare Train/Test Split

In [ ]:
X = df[all_feature_cols].values
y = df['label'].values
feature_col_names = all_feature_cols

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} samples (pos={y_train.sum()}, neg={len(y_train)-y_train.sum()})')
print(f'Test:  {X_test.shape[0]} samples (pos={y_test.sum()}, neg={len(y_test)-y_test.sum()})')

## 4. Train a Shallow Decision Tree (max depth = 4)

In [ ]:
dt = DecisionTreeClassifier(
    max_depth=4,
    random_state=42,
    min_samples_leaf=5,
)
dt.fit(X_train, y_train)

y_pred_train = dt.predict(X_train)
y_pred_test = dt.predict(X_test)

print('=== Decision Tree Performance ===')
print(f'Train accuracy: {accuracy_score(y_train, y_pred_train):.3f}')
print(f'Test accuracy:  {accuracy_score(y_test, y_pred_test):.3f}')
print()
print('Test set classification report:')
print(classification_report(y_test, y_pred_test, target_names=['no_tumor', 'tumor']))

In [ ]:
# Print the tree as text
tree_text = export_text(dt, feature_names=feature_col_names, max_depth=4)
print(tree_text)

## 5. Extract All Root-to-Leaf Rules from the Tree

In [ ]:
def extract_rules(tree, feature_names):
    """
    Walk every root-to-leaf path and return a list of rules.
    Each rule is a dict with:
      - conditions: list of (feature_name, operator, threshold) tuples
      - predicted_class: int
      - train_samples: number of training samples reaching this leaf
      - train_class_counts: per-class counts at the leaf
      - train_purity: fraction of dominant class at the leaf
    """
    tree_ = tree.tree_
    rules = []

    def recurse(node, conditions):
        # Leaf node
        if tree_.children_left[node] == tree_.children_right[node]:
            n_samples = int(tree_.n_node_samples[node])
            # tree_.value stores proportions; multiply by n_samples to get counts
            proportions = tree_.value[node][0]
            class_counts = np.round(proportions * n_samples).astype(int)
            predicted = int(np.argmax(proportions))
            purity = float(proportions[predicted]) if n_samples > 0 else 0.0
            rules.append({
                'conditions': list(conditions),
                'predicted_class': predicted,
                'train_samples': n_samples,
                'train_class_counts': class_counts.tolist(),
                'train_purity': purity,
            })
            return

        feat = feature_names[tree_.feature[node]]
        thresh = tree_.threshold[node]

        # Left child: feature <= threshold
        recurse(tree_.children_left[node], conditions + [(feat, '<=', thresh)])
        # Right child: feature > threshold
        recurse(tree_.children_right[node], conditions + [(feat, '>', thresh)])

    recurse(0, [])
    return rules


rules = extract_rules(dt, feature_col_names)
print(f'Total leaf rules extracted: {len(rules)}')

## 6. Evaluate Each Rule on the Test Set

In [ ]:
def evaluate_rule_on_set(rule, X, y, feature_names):
    """
    Apply a rule's conditions to a dataset and compute coverage and purity.
    Returns (n_matched, n_correct, purity, matched_indices).
    """
    mask = np.ones(len(X), dtype=bool)
    for feat_name, op, thresh in rule['conditions']:
        col_idx = feature_names.index(feat_name)
        if op == '<=':
            mask &= X[:, col_idx] <= thresh
        else:
            mask &= X[:, col_idx] > thresh

    matched_indices = np.where(mask)[0]
    n_matched = mask.sum()
    if n_matched == 0:
        return 0, 0, 0.0, matched_indices

    n_correct = (y[mask] == rule['predicted_class']).sum()
    purity = n_correct / n_matched
    return int(n_matched), int(n_correct), float(purity), matched_indices


# Evaluate every rule on the test set
for rule in rules:
    n_matched, n_correct, purity, _ = evaluate_rule_on_set(
        rule, X_test, y_test, feature_col_names
    )
    rule['test_coverage'] = n_matched
    rule['test_correct'] = n_correct
    rule['test_purity'] = purity

In [ ]:
def format_condition(feat, op, thresh):
    """Format a single condition as a readable string."""
    # For boolean tag features, simplify the display
    if thresh == 0.5 and op == '>':
        return f'{feat} is TRUE'
    elif thresh == 0.5 and op == '<=':
        return f'{feat} is FALSE'
    return f'{feat} {op} {thresh:.4f}'


def format_rule(rule, class_names=['no_tumor', 'tumor']):
    """Format a rule as a readable string."""
    conds = ' AND '.join(format_condition(f, o, t) for f, o, t in rule['conditions'])
    pred = class_names[rule['predicted_class']]
    return f'IF {conds} THEN {pred}'


# Print all rules sorted by test coverage
print('=== All Rules (sorted by test coverage) ===\n')
for i, rule in enumerate(sorted(rules, key=lambda r: r['test_coverage'], reverse=True)):
    print(f'Rule {i+1}: {format_rule(rule)}')
    print(f'  Train: {rule["train_samples"]} samples, purity={rule["train_purity"]:.2%}, '
          f'class counts={rule["train_class_counts"]}')
    print(f'  Test:  {rule["test_coverage"]} samples, purity={rule["test_purity"]:.2%}, '
          f'correct={rule["test_correct"]}/{rule["test_coverage"]}')
    print()

## 7. Select Top 2 Rules Per Class (Highest Coverage, High Purity)

In [ ]:
CLASS_NAMES = ['no_tumor', 'tumor']

selected_rules = {}
for cls in [0, 1]:
    cls_rules = [r for r in rules if r['predicted_class'] == cls]

    # Sort by test purity descending, then by test coverage descending
    cls_rules.sort(key=lambda r: (r['test_purity'], r['test_coverage']), reverse=True)

    selected_rules[cls] = cls_rules[:2]

print('=' * 70)
print('SELECTED RULES (2 per class, highest purity then coverage on test set)')
print('=' * 70)

for cls in [0, 1]:
    print(f'\n--- Class: {CLASS_NAMES[cls]} (label={cls}) ---\n')
    for i, rule in enumerate(selected_rules[cls]):
        print(f'  Rule {i+1}: {format_rule(rule, CLASS_NAMES)}')
        print(f'    Train: {rule["train_samples"]} samples, purity={rule["train_purity"]:.2%}')
        print(f'    Test:  {rule["test_coverage"]} matched, '
              f'{rule["test_correct"]}/{rule["test_coverage"]} correct '
              f'(purity={rule["test_purity"]:.2%})')
        print()

## 8. Evaluate Selected Rules as a Rule-Based Classifier

In [ ]:
all_selected = selected_rules[0] + selected_rules[1]

# For each test sample, check which selected rules fire.
# When multiple rules match, pick the one with highest train purity.
predictions = []
covered_mask = np.zeros(len(X_test), dtype=bool)

for idx in range(len(X_test)):
    x = X_test[idx]
    predicted = None
    best_purity = -1

    for rule in all_selected:
        match = True
        for feat_name, op, thresh in rule['conditions']:
            col_idx = feature_col_names.index(feat_name)
            if op == '<=':
                if x[col_idx] > thresh:
                    match = False
                    break
            else:
                if x[col_idx] <= thresh:
                    match = False
                    break

        if match and rule['train_purity'] > best_purity:
            predicted = rule['predicted_class']
            best_purity = rule['train_purity']
            covered_mask[idx] = True

    predictions.append(predicted)

# Report coverage and accuracy for covered samples
n_covered = covered_mask.sum()
print(f'Coverage: {n_covered}/{len(X_test)} test samples matched by at least one rule '
      f'({n_covered/len(X_test):.1%})')

covered_preds = np.array([p for p in predictions if p is not None])
covered_true = y_test[covered_mask]

if n_covered > 0:
    acc = accuracy_score(covered_true, covered_preds)
    print(f'Accuracy on covered samples: {acc:.3f}')
    print()
    print(classification_report(covered_true, covered_preds, target_names=CLASS_NAMES))

## 9. Visualize the Decision Tree

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    dt,
    feature_names=feature_col_names,
    class_names=CLASS_NAMES,
    filled=True,
    rounded=True,
    ax=ax,
    fontsize=8,
    impurity=True,
    proportion=True,
)
ax.set_title('Decision Tree (max_depth=4) - Brain Tumor Classification', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Summary of Selected Rules for the NSAI Pipeline

In [ ]:
print('Rules that can be used in the NSAI-HUMAINE pipeline:\n')
print('=' * 60)
for cls in [0, 1]:
    for i, rule in enumerate(selected_rules[cls]):
        print(f'{format_rule(rule, CLASS_NAMES)}')
        print(f'  (test purity: {rule["test_purity"]:.0%}, '
              f'test coverage: {rule["test_coverage"]} samples)')
        print()
print('=' * 60)